# MarketingIQ tiny planner — Phase 4: fine-tune Qwen3-0.6B (Unsloth, LoRA, Google Colab)

**What this notebook trains:** a small model that turns a MarketingIQ question (plus, for follow-ups,
the previous analysis plan) into a **compact MarketingIQ plan** — the sparse JSON form of the production
`query_planner.SCHEMA` plan, e.g. `{"intent":"breakdown","metrics":["revenue"],"dimension":"platform"}`.
The model never sees campaign rows or numbers; DuckDB stays the source of truth.

| Setting | Value |
|---|---|
| Base model | `unsloth/Qwen3-0.6B` (verified on Hugging Face and in Unsloth's official Qwen3 (0.6B) notebook, Sept 2026) |
| Method | 16-bit LoRA (not QLoRA): a 0.6B model uses ~1.2 GB in fp16, far below a T4's 15 GB |
| Thinking mode | **OFF** (`enable_thinking=False`, empty `<think></think>` block, same in training and inference) |
| Loss | answer-only: only the plan tokens (+ end-of-turn token) are trained |
| Data | `train.jsonl` (4,524) + `validation.jsonl` (498), SHA-256 checked before training |

**Rules for this phase**
- `eval_frozen.jsonl` (the 429-question final exam) is **not** uploaded, loaded, or used for anything here.
- No API keys, no Groq calls. Nothing is exported to GGUF, nothing is deployed.
- Run the cells **from top to bottom**. Every check stops the notebook with a clear error if something is wrong.

Step-by-step instructions: `experiments/tiny_model/PHASE4_INSTRUCTIONS.md`.

## 0. Configuration
The only cell you might edit. The defaults match `PHASE4_INSTRUCTIONS.md`.

In [ ]:
MODEL_ID = "unsloth/Qwen3-0.6B"   # verified Sept 2026: huggingface.co/unsloth/Qwen3-0.6B

DRIVE_DIR  = "/content/drive/MyDrive/marketingiq_tiny"            # folder you created in Google Drive
BUNDLE_ZIP = f"{DRIVE_DIR}/marketingiq_colab_bundle.zip"           # made by notebooks/make_colab_bundle.py
WORK_DIR   = "/content/marketingiq"                                # bundle is unzipped here (temporary)
OUTPUT_DIR = "/content/outputs"                                    # checkpoints + results (temporary!)
RUN_NAME   = "marketingiq-qwen3-0.6b-lora"

EXPECTED = {  # Phase 3 frozen data (dataset/train_manifest.json)
    "train.jsonl":      ("55452c8e687b9fb78c4ca33b466534cbf63e4039a0be40ba8814c628c9aca007", 4524),
    "validation.jsonl": ("e247278e7ca147e9e2ea8786784d8409a930c2b2b357afa8d54ddbe672c35e73", 498),
}

SEED = 20260929

# LoRA (Unsloth guide: r 16 or 32; alpha = r or 2*r; dropout 0 is the optimized path)
LORA_R = 16
LORA_ALPHA = 32          # 2*r: a stronger update for a narrow task; alpha/r = 2
LORA_DROPOUT = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training (Unsloth guide: lr 2e-4, 1-3 epochs, effective batch 16)
EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH_SIZE = 8           # sequences are short (~100-200 tokens), so 8 per step fits a T4 easily
GRAD_ACCUM = 2           # effective batch 16
EVAL_BATCH_SIZE = 16
EVAL_STEPS = 50
LOGGING_STEPS = 10
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
OPTIMIZER = "adamw_8bit"

GEN_BATCH_SIZE = 32      # validation inference

## 1. Environment Setup
In Colab: **Runtime → Change runtime type → T4 GPU** (or better) before running this.

In [ ]:
import sys, subprocess, platform
print("Python", sys.version.split()[0], "on", platform.platform())

def stop(message):
    raise RuntimeError("\n\n" + "=" * 70 + "\nSTOPPED: " + message + "\n" + "=" * 70)

import torch
if not torch.cuda.is_available():
    stop("No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart and run again.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
props = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_VRAM_GB = props.name, round(props.total_memory / 1024**3, 1)
print(f"GPU: {GPU_NAME} | VRAM: {GPU_VRAM_GB} GB | compute capability {props.major}.{props.minor}")
if GPU_VRAM_GB < 12:
    print("WARNING: less than 12 GB VRAM; lower BATCH_SIZE to 4 and raise GRAD_ACCUM to 4 in section 0.")

Install Unsloth. This is the Colab install cell from Unsloth's official Qwen3 notebooks
(github.com/unslothai/notebooks, checked Sept 2026: `transformers==4.56.2`, `trl==0.22.2`), plus `duckdb`,
which the MarketingIQ validator/executor needs. Takes 1–3 minutes.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install duckdb

In [ ]:
import unsloth  # import before transformers so Unsloth's patches apply
import importlib.metadata as md_
VERSIONS = {p: md_.version(p) for p in ["unsloth", "unsloth_zoo", "torch", "transformers", "trl", "peft",
                                        "datasets", "accelerate", "bitsandbytes", "duckdb", "pandas"]}
VERSIONS["python"] = sys.version.split()[0]
for k, v in VERSIONS.items():
    print(f"{k:14s} {v}")

## 2. Data Upload (Google Drive)
On your laptop, run once (from the repo root):
```
python experiments/tiny_model/notebooks/make_colab_bundle.py
```
It creates `experiments/tiny_model/artifacts/marketingiq_colab_bundle.zip` containing `train.jsonl`,
`validation.jsonl`, `plan_format.py`, `score.py`, and the read-only backend modules + campaign CSV used by
`validate_plan()` / `execute_plan()`. It **never** contains `eval_frozen.jsonl` or secrets.

Upload that zip to Google Drive as **`MyDrive/marketingiq_tiny/marketingiq_colab_bundle.zip`**, then run the next
cell (Colab asks you to allow Drive access).

In [ ]:
import os, zipfile, shutil
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isfile(BUNDLE_ZIP):
    stop(f"Bundle not found at {BUNDLE_ZIP}. Upload marketingiq_colab_bundle.zip to MyDrive/marketingiq_tiny/.")
shutil.rmtree(WORK_DIR, ignore_errors=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z:
    names = z.namelist()
    if any("eval_frozen" in n or "eval_manifest" in n for n in names):
        stop("The bundle contains the frozen evaluation set. It must not be used in Phase 4; rebuild the bundle.")
    z.extractall(WORK_DIR)
print(f"Unzipped {len(names)} files to {WORK_DIR}:")
for n in names:
    print("  ", n)

## 3. Data Integrity
Training only starts on the exact Phase 3 files.

In [ ]:
import hashlib, json
from pathlib import Path

DATA_DIR = Path(WORK_DIR) / "experiments" / "tiny_model" / "dataset"
EVAL_DIR = Path(WORK_DIR) / "experiments" / "tiny_model" / "evaluation"

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

if list(Path(WORK_DIR).rglob("eval_frozen*")):
    stop("eval_frozen.jsonl is present in the working directory. Remove it; it is not used in Phase 4.")

DATA_HASHES, problems = {}, []
for name, (want_hash, want_rows) in EXPECTED.items():
    path = DATA_DIR / name
    if not path.is_file():
        stop(f"{name} is missing from the bundle.")
    got_hash = sha256_file(path)
    rows = sum(1 for line in path.open(encoding="utf-8") if line.strip())
    DATA_HASHES[name] = got_hash
    print(f"{name:17s} SHA-256 {got_hash}  {'OK' if got_hash == want_hash else 'MISMATCH'}")
    print(f"{'':17s} rows: {rows} (expected {want_rows})  {'OK' if rows == want_rows else 'MISMATCH'}")
    if got_hash != want_hash:
        problems.append(f"{name}: SHA-256 {got_hash} != expected {want_hash}")
    if rows != want_rows:
        problems.append(f"{name}: {rows} rows != expected {want_rows}")
if problems:
    stop("Dataset integrity check failed - DO NOT TRAIN.\n" + "\n".join(problems))
print("\nIntegrity OK: training data is the frozen Phase 3 data.")

## 4. Load Dataset
Real Phase 3 record structure: `id, template_id, category, source, style, question, previous, plan`.
`previous` is `null` or a compact plan; `plan` is the compact gold plan. Metadata is kept for per-category results.

In [ ]:
import collections

for d in (DATA_DIR, EVAL_DIR):
    if str(d) not in sys.path:
        sys.path.insert(0, str(d))
import plan_format as pf   # Phase 1 prompt + compact-plan contract (reused, not re-implemented)

FIELDS = ["id", "template_id", "category", "source", "style", "question", "previous", "plan"]

def load_jsonl(path):
    rows = [json.loads(line) for line in Path(path).open(encoding="utf-8") if line.strip()]
    for r in rows:
        if list(r) != FIELDS:
            stop(f"{path}: unexpected fields {list(r)} in {r.get('id')}")
    return rows

train_rows = load_jsonl(DATA_DIR / "train.jsonl")
val_rows = load_jsonl(DATA_DIR / "validation.jsonl")

# The read-only MarketingIQ backend from the bundle: plan_format needs its SCHEMA (compact-plan key order) and
# validate_plan() needs the dataset's real values. No LLM, no network, nothing is written.
try:
    pf.backend()
except Exception as e:
    stop(f"MarketingIQ backend modules could not load ({type(e).__name__}: {e}). Re-run section 1 (duckdb install).")
import score as scorer   # Phase 2 scorer: to_canonical -> validate_plan -> execute_plan on DuckDB

# Execution smoke test: gold plans must execute on the bundled data. If this fails, execution-based metrics are
# reported as NOT RUN in section 12 (never guessed).
EXEC_OK, EXEC_ERROR = True, None
try:
    for r in val_rows[:5]:
        s = scorer.score(r["plan"], json.dumps(r["plan"]), r["question"])
        assert s["exact"] and s["execution_match"], f"gold plan {r['id']} does not match itself"
except Exception as e:
    EXEC_OK, EXEC_ERROR = False, f"{type(e).__name__}: {e}"
    print("WARNING: plan execution is unavailable; execution metrics will be reported as NOT RUN:", EXEC_ERROR)
print("validate_plan available: True | DuckDB plan execution available:", EXEC_OK)

for name, rows in (("train", train_rows), ("validation", val_rows)):
    print(f"{name}: {len(rows)} rows | follow-ups (previous != null): {sum(r['previous'] is not None for r in rows)}")
    print("   categories:", dict(collections.Counter(r["category"] for r in rows).most_common()))
    print("   intents:   ", dict(collections.Counter(r["plan"]["intent"] for r in rows).most_common()))
    # targets must already be compact (defaults omitted, SCHEMA key order)
    bad = [r["id"] for r in rows if pf.to_sparse(r["plan"]) != r["plan"]
           or (r["previous"] is not None and pf.to_sparse(r["previous"]) != r["previous"])]
    if bad:
        stop(f"{len(bad)} {name} plans are not in compact form, e.g. {bad[:5]}")

overlap = {r["question"].strip().lower() for r in train_rows} & {r["question"].strip().lower() for r in val_rows}
print("train/validation identical questions:", len(overlap))
print("\nExample record:\n", json.dumps(next(r for r in val_rows if r["previous"]), indent=1))

## 5. Training Prompt Format
Reuses Phase 1 `plan_format.messages()` exactly:
- **system**: `plan_format.SYSTEM_PROMPT` (short and fixed — it is prefilled on a CPU later),
- **user**: the question; for follow-ups, `plan_format.user_prompt()` prepends the previous compact plan in the same
  wording `query_planner.answer()` sends Groq (no full conversation history),
- **assistant (target)**: the compact plan, minified JSON. Nothing else: no data, no explanation.

Thinking is disabled with Qwen3's `enable_thinking=False` chat-template switch (current Qwen/Unsloth API). The prompt
ends with the empty `<think>

</think>

` block, so the model goes straight to the plan. Training text is
built as `prompt + plan + <|im_end|>`, and the cell checks it is byte-identical to the template's own rendering of the
full conversation, so training and inference prompts cannot drift.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)   # tokenizer files only (a few MB), no weights yet
EOS = "<|im_end|>"
THINK_OFF = "<think>\n\n</think>\n\n"
if tokenizer.eos_token != EOS:
    stop(f"Unexpected eos token {tokenizer.eos_token!r}; expected {EOS!r} for Qwen3 chat.")

def format_row(row, tok):
    """prompt (what inference sends), completion (the compact plan), text (training sequence)."""
    prompt = tok.apply_chat_template(pf.messages(row["question"], row["previous"]), tokenize=False,
                                     add_generation_prompt=True, enable_thinking=False)
    full_msgs = pf.messages(row["question"], row["previous"], row["plan"])
    completion = full_msgs[-1]["content"]
    full = tok.apply_chat_template(full_msgs, tokenize=False, enable_thinking=False)
    if not prompt.endswith("<|im_start|>assistant\n" + THINK_OFF):
        stop(f"{row['id']}: prompt does not end with the thinking-OFF assistant header")
    if full != prompt + completion + EOS + "\n":
        stop(f"{row['id']}: training rendering differs from inference prompt + plan:\n{full!r}")
    if "<think>" in completion or json.loads(completion) != row["plan"]:
        stop(f"{row['id']}: completion is not exactly the compact gold plan")
    return {"prompt": prompt, "completion": completion, "text": prompt + completion + EOS}

train_fmt = [format_row(r, tokenizer) for r in train_rows]
val_fmt = [format_row(r, tokenizer) for r in val_rows]
print(f"Formatted {len(train_fmt)} train + {len(val_fmt)} validation examples; all prompts match inference format.\n")
first = train_rows.index(next(r for r in train_rows if r["previous"] is None))
follow = train_rows.index(next(r for r in train_rows if r["previous"] is not None))
for i in (first, follow):
    print("-" * 30, train_rows[i]["id"], train_rows[i]["category"], "-" * 30)
    print(train_fmt[i]["text"], "\n")

## 6. Tokenization Analysis
Measured on the real formatted examples (train + validation) before choosing `max_seq_length`.

In [ ]:
import numpy as np

def lengths(texts):
    return np.array([len(ids) for ids in tokenizer(texts, add_special_tokens=False)["input_ids"]])

stats = {}
for split, fmt in (("train", train_fmt), ("validation", val_fmt)):
    for part in ("text", "prompt", "completion"):
        stats[(split, part)] = lengths([f[part] for f in fmt])

print(f"{'split':11s} {'part':11s} {'max':>5s} {'p50':>5s} {'p90':>5s} {'p95':>5s} {'p99':>5s} {'mean':>6s}")
for (split, part), L in stats.items():
    p = np.percentile(L, [50, 90, 95, 99])
    print(f"{split:11s} {part:11s} {L.max():5d} {p[0]:5.0f} {p[1]:5.0f} {p[2]:5.0f} {p[3]:5.0f} {L.mean():6.1f}")

longest = max(stats[("train", "text")].max(), stats[("validation", "text")].max())
MAX_SEQ_LENGTH = int(np.ceil((longest + 32) / 64) * 64)       # longest example + margin, multiple of 64
longest_plan = max(stats[("train", "completion")].max(), stats[("validation", "completion")].max())
MAX_NEW_TOKENS = int(longest_plan + 32)                        # generation budget for validation
TOKEN_STATS = {f"{s}_{p}": {"max": int(L.max()), "p50": float(np.percentile(L, 50)), "p90": float(np.percentile(L, 90)),
                            "p95": float(np.percentile(L, 95)), "p99": float(np.percentile(L, 99))}
               for (s, p), L in stats.items()}
assert all(L.max() <= MAX_SEQ_LENGTH for (s, p), L in stats.items() if p == "text"), "an example would be truncated"
print(f"\nLongest full example: {longest} tokens -> selected MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH} (no example truncated)")
print(f"Longest gold plan: {longest_plan} tokens -> MAX_NEW_TOKENS = {MAX_NEW_TOKENS}")

## 7. Load Qwen3-0.6B (16-bit)
`load_in_4bit=False`: plain 16-bit LoRA. Unsloth's guide: 16-bit LoRA is slightly faster and more accurate than
QLoRA; QLoRA only saves memory, which a 0.6B model on a T4 does not need. `dtype=None` lets Unsloth pick fp16 on a
T4 (no bf16 support) and bf16 on newer GPUs.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import set_seed
set_seed(SEED)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,
)
# The Unsloth tokenizer must render exactly what section 5 checked (same template, same thinking-OFF format).
drift = [r["id"] for r, f in zip(train_rows + val_rows, train_fmt + val_fmt) if format_row(r, tokenizer) != f]
if drift:
    stop(f"Unsloth's tokenizer renders {len(drift)} examples differently from section 5, e.g. {drift[:3]}")
print(f"Model dtype: {model.dtype} | GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("Chat template check: identical for all", len(train_rows) + len(val_rows), "examples")

## 8. LoRA Configuration

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    target_modules = TARGET_MODULES,   # all attention + MLP linear projections of Qwen3
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)
model.print_trainable_parameters()

## 9. Training Configuration
2 epochs, lr 2e-4 (linear decay, 3% warmup), effective batch 16 (8 × 2 accumulation) → ~566 optimizer steps.
Validation loss every 50 steps; a checkpoint at the end of each epoch.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

tokenizer.padding_side = "right"   # training; switched to "left" for batched generation in section 12
train_ds = Dataset.from_list([{"text": f["text"]} for f in train_fmt])
val_ds = Dataset.from_list([{"text": f["text"]} for f in val_fmt])
CKPT_DIR = f"{OUTPUT_DIR}/checkpoints"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,     # validation.jsonl only - never eval_frozen.jsonl
    args = SFTConfig(
        output_dir = CKPT_DIR,
        dataset_text_field = "text",
        max_length = MAX_SEQ_LENGTH,
        packing = False,
        num_train_epochs = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        per_device_eval_batch_size = EVAL_BATCH_SIZE,
        learning_rate = LEARNING_RATE,
        lr_scheduler_type = "linear",
        warmup_ratio = WARMUP_RATIO,
        weight_decay = WEIGHT_DECAY,
        optim = OPTIMIZER,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = LOGGING_STEPS,
        eval_strategy = "steps",
        eval_steps = EVAL_STEPS,
        save_strategy = "epoch",
        save_total_limit = EPOCHS,
        seed = SEED,
        data_seed = SEED,
        report_to = "none",
    ),
)

## 10. Answer-Only Loss
Unsloth's `train_on_responses_only` sets every label before the assistant's answer to -100, so the loss only covers
the compact plan and its `<|im_end|>`. The response marker includes the empty think block, so the fixed
`<think>

</think>

` (part of the prompt at inference) is not trained either.

The check below then verifies **every** train and validation row: the trained tokens must decode to exactly
`<gold compact plan><|im_end|>`, and no system/question/previous-plan token may carry loss. It also inspects a real
collated batch. Any failure stops the notebook.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

INSTRUCTION_PART = "<|im_start|>user\n"
RESPONSE_PART = "<|im_start|>assistant\n" + THINK_OFF
for attr in ("_unsloth_input_part", "_unsloth_output_part"):   # use our markers, not a preset
    if hasattr(tokenizer, attr):
        print(f"(removing preset {attr} = {getattr(tokenizer, attr)!r})")
        delattr(tokenizer, attr)
trainer = train_on_responses_only(trainer, instruction_part=INSTRUCTION_PART, response_part=RESPONSE_PART)

In [ ]:
def check_masking(dataset, fmt, split):
    """Every row: loss tokens == tokens of '<compact plan><|im_end|>', prompt tokens all masked."""
    if len(dataset) != len(fmt):
        stop(f"{split}: {len(dataset)} rows after masking, expected {len(fmt)} (rows were dropped)")
    trained_total = all_total = 0
    for i, (row, f) in enumerate(zip(dataset, fmt)):
        ids, labels = list(row["input_ids"]), list(row["labels"])
        if tokenizer.decode(ids) != f["text"]:
            stop(f"{split} row {i}: tokenized text differs from the formatted example (truncated or altered)")
        trained = [t for t, l in zip(ids, labels) if l != -100]
        if any(l != -100 and l != t for t, l in zip(ids, labels)):
            stop(f"{split} row {i}: labels differ from input_ids")
        first = next((k for k, l in enumerate(labels) if l != -100), None)
        if first is None:
            stop(f"{split} row {i}: no token carries loss")
        if any(l == -100 for l in labels[first:]):
            stop(f"{split} row {i}: trained tokens are not one contiguous answer span")
        if tokenizer.decode(ids[:first]) != f["prompt"]:
            stop(f"{split} row {i}: masked part is not exactly the prompt")
        if tokenizer.decode(trained) != f["completion"] + EOS:
            stop(f"{split} row {i}: trained tokens are {tokenizer.decode(trained)!r}, expected the plan + {EOS}")
        trained_total += len(trained)
        all_total += len(ids)
    print(f"{split}: {len(dataset)} rows OK | loss on {trained_total}/{all_total} tokens "
          f"({100 * trained_total / all_total:.1f}%) - prompt tokens never trained")
    return trained_total / all_total

MASK_FRACTION = {"train": check_masking(trainer.train_dataset, train_fmt, "train"),
                 "validation": check_masking(trainer.eval_dataset, val_fmt, "validation")}
if MASK_FRACTION["train"] > 0.8:
    stop("Almost every token is trained - answer-only masking is not active.")

# What the loss actually sees: one real collated training batch.
batch = trainer.data_collator([{k: trainer.train_dataset[i][k] for k in ("input_ids", "attention_mask", "labels")}
                               for i in range(4)])
for ids, labels in zip(batch["input_ids"].tolist(), batch["labels"].tolist()):
    answer = tokenizer.decode([t for t, l in zip(ids, labels) if l != -100])
    if not (answer.startswith("{") and answer.endswith(EOS)) or "<|im_start|>" in answer:
        stop(f"Collated batch trains on {answer!r}")
print("Collated batch: labels cover only the plan + <|im_end|>.\n")

# Diagnostic: one follow-up example. [LOSS] = token contributes to the loss, ...... = masked (label -100).
ex = trainer.train_dataset[follow]
ids, labels = list(ex["input_ids"]), list(ex["labels"])
first = next(k for k, l in enumerate(labels) if l != -100)
print(f"Example {train_rows[follow]['id']}: {len(ids)} tokens, {first} masked, {len(ids) - first} trained")
print(f"  ......  tokens 0-{first - 9}: system prompt + previous plan + question (all masked)")
for t, l in list(zip(ids, labels))[first - 8:]:
    print(f"  {'[LOSS]' if l != -100 else '......'}  {tokenizer.decode([t])!r}")

## 11. Training
---
# ▶ START GPU TRAINING HERE
Takes roughly 10–20 minutes on a free T4 (2 epochs, ~566 steps). Keep the browser tab open.
The table shows training loss every 10 steps and validation loss every 50 steps.

In [ ]:
import time
t0 = time.time()
trainer_stats = trainer.train()
TRAIN_MINUTES = (time.time() - t0) / 60
eval_metrics = trainer.evaluate()   # final validation loss of the finished model
print(f"\nTraining time: {TRAIN_MINUTES:.1f} min | peak GPU memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print(f"Average training loss: {trainer_stats.training_loss:.4f} | final validation loss: {eval_metrics['eval_loss']:.4f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
LOG_HISTORY = trainer.state.log_history
log = pd.DataFrame(LOG_HISTORY)
cols = [c for c in ["step", "epoch", "loss", "eval_loss", "learning_rate"] if c in log]
print(log[cols].to_string(index=False, max_rows=40))
train_log, eval_log = log.dropna(subset=["loss"]), log.dropna(subset=["eval_loss"])
FINAL_TRAIN_LOSS = float(train_log["loss"].iloc[-1])
FINAL_EVAL_LOSS = float(eval_metrics["eval_loss"])
plt.figure(figsize=(7, 3.5))
plt.plot(train_log["step"], train_log["loss"], label="training loss")
plt.plot(eval_log["step"], eval_log["eval_loss"], marker="o", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.yscale("log"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 12. Validation (structured-plan quality, `validation.jsonl` only)
Loss alone does not say whether plans are right. Each validation question is answered by the model (greedy decoding,
thinking OFF), then:

`model output → parse JSON → plan_format.to_canonical() → validate_plan() → execute_plan() on DuckDB`

using Phase 2 `evaluation/score.py` (the same scorer the final benchmark uses):

| Metric | Meaning |
|---|---|
| parse rate | output is a JSON object |
| validate_plan pass rate | production `validate_plan()` accepts the plan (a not-answerable refusal counts as accepted) |
| exact plan match (strict) | predicted compact JSON == gold compact JSON |
| exact plan match (semantic) | `score.py` `exact`: every plan field means the same after validation |
| execution match | `score.py` `execution_match`: identical executed answer on the bundled campaign data (real DuckDB run) |

In [ ]:
def generate_plans(prompts, batch_size=GEN_BATCH_SIZE):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    outputs = []
    for i in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True,
                        add_special_tokens=False).to("cuda")
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 eos_token_id=tokenizer.convert_tokens_to_ids(EOS),
                                 pad_token_id=tokenizer.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        outputs += [t.strip() for t in tokenizer.batch_decode(new, skip_special_tokens=True)]
        print(f"\r{len(outputs)}/{len(prompts)} generated", end="")
    print()
    return outputs

t0 = time.time()
raw_outputs = generate_plans([f["prompt"] for f in val_fmt])
GEN_SECONDS = time.time() - t0
print(f"{len(raw_outputs)} validation plans in {GEN_SECONDS:.0f}s (GPU batch inference; not a CPU latency number)")

In [ ]:
import pandas as pd

def parse_plan(text):
    try:
        plan = json.loads(text)
    except (ValueError, TypeError):
        return None
    return plan if isinstance(plan, dict) else None

results = []
for row, out in zip(val_rows, raw_outputs):
    pred = parse_plan(out)
    res = {"id": row["id"], "template_id": row["template_id"], "category": row["category"], "source": row["source"],
           "style": row["style"], "question": row["question"], "previous": row["previous"], "gold": row["plan"],
           "raw_output": out, "parsed": pred is not None, "exact_strict": pred == row["plan"]}
    if EXEC_OK:
        try:
            s = scorer.score(row["plan"], out, row["question"])
        except Exception as e:   # a crash here is a setup problem, not a model error
            stop(f"score.py failed on {row['id']}: {type(e).__name__}: {e}")
        res.update(validate_plan_ok=s["validator_accepted"], validator_error=s["error"],
                   exact_semantic=s["exact"], execution_match=s["execution_match"])
    else:                        # validate_plan only (same call score.py makes), no execution
        _, _, err = scorer._validated(out, row["question"])
        res.update(validate_plan_ok=err is None, validator_error=err)
    results.append(res)

def rate(key, rows=results):
    return sum(r[key] for r in rows) / len(rows)

NOT_RUN = f"NOT RUN (plan execution unavailable: {EXEC_ERROR})"
METRICS = {"validation_examples": len(results),
           "parse_rate": rate("parsed"),
           "validate_plan_rate": rate("validate_plan_ok"),
           "exact_match_strict": rate("exact_strict"),
           "exact_match_semantic": rate("exact_semantic") if EXEC_OK else NOT_RUN,
           "execution_match": rate("execution_match") if EXEC_OK else NOT_RUN}
print("VALIDATION RESULTS (validation.jsonl, 498 examples)")
for k, v in METRICS.items():
    print(f"  {k:22s} {v:.2%}" if isinstance(v, float) else f"  {k:22s} {v}")

by = pd.DataFrame(results)
cols = ["parsed", "validate_plan_ok", "exact_strict"] + (["exact_semantic", "execution_match"] if EXEC_OK else [])
per_cat = by.groupby("category")[cols].mean().mul(100).round(1)
per_cat.insert(0, "n", by.groupby("category").size())
print("\nPer category (%):\n", per_cat.sort_values("n", ascending=False).to_string())
print("\nFollow-ups vs single questions (%):\n",
      by.assign(follow_up=by["previous"].notna()).groupby("follow_up")[cols].mean().mul(100).round(1).to_string())

## 13. Sample Predictions
A few misses (if any) and a few correct predictions, including a follow-up.

In [ ]:
def show(r):
    print("Question:        ", r["question"])
    if r["previous"] is not None:
        print("Previous plan:   ", json.dumps(r["previous"], separators=(",", ":")))
    print("Gold plan:       ", json.dumps(r["gold"], separators=(",", ":")))
    print("Predicted plan:  ", r["raw_output"])
    status = "accepted" if r["validate_plan_ok"] else f"REJECTED ({r['validator_error']})"
    extra = f" | semantic match: {r['exact_semantic']} | execution match: {r['execution_match']}" if EXEC_OK else ""
    print(f"Exact match: {r['exact_strict']}{extra} | validate_plan: {status}")
    print("-" * 100)

key = "execution_match" if EXEC_OK else "exact_strict"
misses = [r for r in results if not r[key]]
hits = [r for r in results if r[key]]
sample = misses[:5] + [r for r in hits if r["previous"] is not None][:2] + [r for r in hits if r["previous"] is None][:3]
print(f"{len(misses)} misses out of {len(results)} by {key}. Showing {len(sample)} examples.\n")
for r in sample:
    show(r)

## 14. Save the Trained LoRA
Saves the LoRA adapter (weights + adapter config), tokenizer, the prompt contract needed for inference later,
the training log, the validation predictions, and `training_report.json`. No merging, no GGUF (later phases).

In [ ]:
import datetime
RUN_DIR = Path(OUTPUT_DIR) / RUN_NAME
ADAPTER_DIR = RUN_DIR / "adapter"
RUN_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

PROMPT_CONTRACT = {"system_prompt": pf.SYSTEM_PROMPT, "chat_template_kwargs": {"enable_thinking": False},
                   "add_generation_prompt": True, "stop_token": EOS, "output": "compact plan JSON (plan_format.to_sparse)",
                   "decode_with": "plan_format.to_canonical -> query_planner.validate_plan",
                   "max_seq_length": MAX_SEQ_LENGTH, "max_new_tokens": MAX_NEW_TOKENS}
REPORT = {
    "phase": 4, "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "model_id": MODEL_ID, "training_method": "LoRA (16-bit base, Unsloth)", "thinking_mode": "OFF",
    "answer_only_loss": True, "loss_token_fraction": MASK_FRACTION,
    "training_examples": len(train_rows), "validation_examples": len(val_rows),
    "frozen_eval_used": False, "dataset_sha256": DATA_HASHES,
    "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "lr_scheduler": "linear", "warmup_ratio": WARMUP_RATIO,
    "batch_size": BATCH_SIZE, "gradient_accumulation_steps": GRAD_ACCUM, "effective_batch_size": BATCH_SIZE * GRAD_ACCUM,
    "optimizer": OPTIMIZER, "weight_decay": WEIGHT_DECAY,
    "lora_rank": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT, "target_modules": TARGET_MODULES,
    "max_seq_length": MAX_SEQ_LENGTH, "max_new_tokens": MAX_NEW_TOKENS, "token_length_stats": TOKEN_STATS,
    "seed": SEED, "total_steps": trainer.state.global_step, "training_minutes": round(TRAIN_MINUTES, 1),
    "final_training_loss": FINAL_TRAIN_LOSS, "average_training_loss": float(trainer_stats.training_loss),
    "final_validation_loss": FINAL_EVAL_LOSS,
    "validation_parse_rate": METRICS["parse_rate"],
    "validation_validate_plan_rate": METRICS["validate_plan_rate"],
    "validation_exact_match_rate": METRICS["exact_match_strict"],
    "validation_exact_match_semantic_rate": METRICS["exact_match_semantic"],
    "validation_execution_match_rate": METRICS["execution_match"],
    "validation_per_category_pct": json.loads(per_cat.to_json(orient="index")),
    "execution_metrics_available": EXEC_OK,
    "checkpoint_used": "final model after the last epoch (no checkpoint selection)",
    "gpu": {"name": GPU_NAME, "vram_gb": GPU_VRAM_GB}, "versions": VERSIONS,
    "prompt_contract": PROMPT_CONTRACT, "groq_calls": 0,
}
(RUN_DIR / "training_report.json").write_text(json.dumps(REPORT, indent=2))
(ADAPTER_DIR / "marketingiq_prompt_contract.json").write_text(json.dumps(PROMPT_CONTRACT, indent=2))
(RUN_DIR / "training_log_history.json").write_text(json.dumps(LOG_HISTORY, indent=1))
with (RUN_DIR / "validation_predictions.jsonl").open("w", encoding="utf-8") as fh:
    for r in results:
        fh.write(json.dumps(r) + "\n")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(RUN_DIR)}")

## 15. Persist Results to Google Drive
**Colab deletes `/content` when the session ends.** This copies the adapter, report, logs, predictions (and the
epoch checkpoints) to `MyDrive/marketingiq_tiny/runs/<timestamp>_marketingiq-qwen3-0.6b-lora/` and verifies every
copied file.

In [ ]:
COPY_CHECKPOINTS = True
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
DRIVE_RUN = Path(DRIVE_DIR) / "runs" / f"{stamp}_{RUN_NAME}"
shutil.copytree(RUN_DIR, DRIVE_RUN)
if COPY_CHECKPOINTS and Path(CKPT_DIR).is_dir():
    shutil.copytree(CKPT_DIR, DRIVE_RUN / "checkpoints")
bad = [p for p in RUN_DIR.rglob("*") if p.is_file()
       and sha256_file(p) != sha256_file(DRIVE_RUN / p.relative_to(RUN_DIR))]
if bad:
    stop(f"Copy to Drive failed for {bad}")
print("Saved to Google Drive:", DRIVE_RUN)
print("Verified", sum(1 for p in RUN_DIR.rglob('*') if p.is_file()), "files (SHA-256 identical).")
print("\nOptional: download the report to this computer too:")
print("  from google.colab import files; files.download(str(DRIVE_RUN / 'training_report.json'))")

## Phase 4 ends here
Result: a LoRA adapter + validation metrics + `training_report.json`, saved to Google Drive.

**Not in this phase:** running `eval_frozen.jsonl`, GGUF export/quantization, llama.cpp/Ollama, laptop latency
benchmarks, FastAPI integration, deployment. The frozen 429-question set is only opened for the final benchmark.